# Incident Download (Planet + GEE + SAR + DEM)

Downloads missing incident rasters using Planet orders and GEE fallback rules.
Uploads into `raw_images/raw_incidents/incident_{ID}/` in Hugging Face.

In [1]:
import os
import re
import glob
import shutil
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
import rasterio
from rasterio.merge import merge as rio_merge
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, CommitOperationAdd

# ----------------------------
# User configuration
# ----------------------------
INPUT_CSV = '/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv'
START_IDX = 0
END_IDX = 0

PRE_DAYS = 180        # configurable, default 6 months
POST_DAYS = 30
CLOUD_MAX_AOI = 5
MAX_AOI_DEG = 0.1

ORDERS_URL = 'https://api.planet.com/compute/ops/orders/v2'
WANTED_STATES = {'success', 'partial'}
REQUEST_TIMEOUT = 300
# Each Planet order's results include multiple asset types per scene (per-scene metadata.json,
# udm2 usable-data-mask clip, AnalyticMS metadata xml, plus an order-level manifest.json) - only
# the analytic surface-reflectance GeoTIFF is the actual imagery we want to download/mosaic.
ANALYTIC_SR_SUFFIX = '_3b_analyticms_sr_clip.tif'

MAX_WORKERS = 2          # incidents downloaded/processed concurrently - tune for Kaggle CPU/network limits
UPLOAD_BATCH_SIZE = 25   # number of ready .tif files to accumulate before flushing one batched HF commit

GEE_PROJECT = 'landslide-identification-nepal'
GEE_SERVICE_ACCOUNT = 'kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com'
GEE_KEY_PATH = '/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json'

HF_REPO_ID = 'sasudo2/landslides'
HF_REPO_TYPE = 'dataset'
HF_REVISION = 'main'
HF_RAW_ROOT = 'raw_images/raw_incidents'
HF_DOWNLOAD_LOG = 'raw_images/download_log.csv'

WORK_DIR = '/kaggle/working/raw_incidents'
os.makedirs(WORK_DIR, exist_ok=True)

In [2]:
def clamp_aoi(min_lon, min_lat, max_lon, max_lat, max_deg=MAX_AOI_DEG):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= max_deg and lat_span <= max_deg:
        return float(min_lon), float(min_lat), float(max_lon), float(max_lat)
    cx = (min_lon + max_lon) / 2.0
    cy = (min_lat + max_lat) / 2.0
    half = max_deg / 2.0
    return float(cx - half), float(cy - half), float(cx + half), float(cy + half)

def make_planet_session(api_key):
    s = requests.Session()
    s.auth = (api_key, '')
    retries = Retry(total=5, backoff_factor=2, status_forcelist=[429,500,502,503,504],
                    allowed_methods=frozenset(['GET']), respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=retries))
    return s

def list_orders(session):
    orders = []
    url = ORDERS_URL
    while url:
        r = session.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()
        orders.extend(data.get('orders', []))
        url = data.get('_links', {}).get('_next')
    return orders

def extract_order_results(session, order):
    results = order.get('_links', {}).get('results')
    if results:
        return results
    oid = order.get('id')
    r = session.get(f'{ORDERS_URL}/{oid}', timeout=120)
    r.raise_for_status()
    return r.json().get('_links', {}).get('results', []) or []

def extract_order_asset_links(session, order, suffix):
    # Order results mix imagery with per-scene metadata.json/.xml, udm2 masks, and an
    # order-level manifest.json - keep only the files whose delivered name ends with `suffix`
    # (e.g. the analytic SR GeoTIFF) so we never try to mosaic a non-raster/wrong asset.
    links = []
    for r in extract_order_results(session, order):
        name = (r.get('name') or '').lower()
        loc = r.get('location')
        if loc and name.endswith(suffix.lower()):
            links.append(loc)
    return links

def download_file(session, url, out_path):
    tmp = out_path + '.part'
    with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=(1 << 20)):
                if chunk:
                    f.write(chunk)
    os.replace(tmp, out_path)

def mosaic_geotiffs(src_paths, dst_path):
    srcs = [rasterio.open(p) for p in src_paths]
    try:
        mosaic, out_transform = rio_merge(srcs)
        out_meta = srcs[0].meta.copy()
        out_meta.update({
            'height': mosaic.shape[1],
            'width': mosaic.shape[2],
            'transform': out_transform,
            'count': mosaic.shape[0],
        })
        with rasterio.open(dst_path, 'w', **out_meta) as dst:
            dst.write(mosaic)
    finally:
        for s in srcs:
            s.close()

def get_hf_existing_incident_files(api):
    existing = {}
    try:
        for f in api.list_repo_files(HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION):
            m = re.match(r'^raw_images/raw_incidents/incident_(\d+)/(.*)$', f)
            if m:
                inc_id = int(m.group(1))
                existing.setdefault(inc_id, set()).add(m.group(2))
    except Exception as e:
        print(f'Warning: could not list HF files: {e}')
    return existing

In [3]:
# Auth
secrets = UserSecretsClient()
planet_api_key = secrets.get_secret('Lokesh_planet')
hf_token = secrets.get_secret('huggingface_token')

planet = make_planet_session(planet_api_key)
hf_api = HfApi(token=hf_token)

# GEE init - prefer the service-account key JSON stored as a Kaggle secret (works even
# when the gee-key dataset isn't attached/mounted in this notebook run); fall back to the
# GEE_KEY_PATH file if no such secret is configured.
gee_available = False
try:
    import ee
    gee_key_json = None
    try:
        gee_key_json = secrets.get_secret('gee_json_key')
    except Exception:
        gee_key_json = None
    if gee_key_json:
        creds = ee.ServiceAccountCredentials(GEE_SERVICE_ACCOUNT, key_data=gee_key_json)
    else:
        creds = ee.ServiceAccountCredentials(GEE_SERVICE_ACCOUNT, GEE_KEY_PATH)
    ee.Initialize(creds, project=GEE_PROJECT)
    gee_available = True
    print('GEE initialized')
except Exception as e:
    print(f'GEE unavailable: {e}')

# Load incidents
df = pd.read_csv(INPUT_CSV)
if START_IDX == 0 and END_IDX == 0:
    df_sel = df.copy()
else:
    df_sel = df.iloc[START_IDX:END_IDX].copy()
df_sel['incident_on'] = pd.to_datetime(df_sel['incident_on'], dayfirst=True)
print(f'Selected incidents: {len(df_sel)}')

existing_files = get_hf_existing_incident_files(hf_api)
all_orders = list_orders(planet)
state_counts = {}
for o in all_orders:
    s = o.get('state')
    state_counts[s] = state_counts.get(s, 0) + 1
print(f'Planet order state breakdown (total {len(all_orders)}): {state_counts}')
orders = [o for o in all_orders if o.get('state') in WANTED_STATES]
print(f'Planet orders in downloadable states: {len(orders)}')

orders_by_name = {o.get('name', ''): o for o in orders}

# Only look at incidents that actually have a ready Planet order (after and/or before) -
# scan Planet.com first instead of looping over every incident in the CSV on every run.
order_name_re = re.compile(r'^incident_(\d+)_planet_(?:after|before)$')
incident_ids_with_orders = {int(m.group(1)) for name in orders_by_name if (m := order_name_re.match(name))}
df_sel = df_sel[df_sel['id'].astype(int).isin(incident_ids_with_orders)].copy()
print(f'Incidents with a ready Planet order: {len(df_sel)}')

/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


GEE initialized
Selected incidents: 4138
Planet orders in downloadable states: 20
Incidents with a ready Planet order: 20


In [4]:
def gee_cloud_helpers():
    def mask_s2_clouds(image):
        scl = image.select('SCL')
        clean = (scl.eq(2).bitwiseOr(scl.eq(4)).bitwiseOr(scl.eq(5)).bitwiseOr(scl.eq(6)).bitwiseOr(scl.eq(7)).bitwiseOr(scl.eq(11)))
        return image.updateMask(clean)

    def add_aoi_cloud(img, aoi):
        scl = img.select('SCL')
        cloud = (scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)))
        stats = cloud.reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi, scale=60, maxPixels=1e9)
        frac = stats.get('SCL')
        pct = ee.Algorithms.If(frac, ee.Number(frac).multiply(100), ee.Number(100))
        return img.set('aoi_cloud', pct)

    return mask_s2_clouds, add_aoi_cloud

def gee_download_url(image, aoi, scale=10):
    return image.getDownloadURL({'scale': scale, 'region': aoi, 'format': 'GeoTIFF', 'crs': 'EPSG:4326'})

def gee_download_to_path(image, aoi, path, scale=10):
    url = gee_download_url(image, aoi, scale=scale)
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    with open(path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)


In [5]:
def process_incident(row):
    """Download Planet (after/before) + GEE/SAR/DEM assets for one incident.
    Runs inside a worker thread - must not mutate shared state (existing_files,
    orders_by_name, download_records, etc. are only read here).
    Returns (inc_id, inc_dir, tif_files) where tif_files is the list of locally
    downloaded .tif paths (empty if nothing was available to download yet)."""
    inc_id = int(row['id'])
    inc_dir = os.path.join(WORK_DIR, f'incident_{inc_id}')
    os.makedirs(inc_dir, exist_ok=True)

    have = existing_files.get(inc_id, set())
    before_name = f'incident_{inc_id}_planet_before'
    after_name = f'incident_{inc_id}_planet_after'
    after_ready = after_name in orders_by_name
    after_have = f'incident_{inc_id}_after.tif' in have

    min_lon, min_lat, max_lon, max_lat = clamp_aoi(row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    incident_date = pd.to_datetime(row['incident_on'], dayfirst=True)

    # Planet AFTER download + mosaic
    if not after_have and after_ready:
        try:
            order = orders_by_name[after_name]
            links = extract_order_asset_links(planet, order, ANALYTIC_SR_SUFFIX)
            parts = []
            for i, link in enumerate(links, 1):
                p = os.path.join(inc_dir, f'_after_part_{i}.tif')
                download_file(planet, link, p)
                parts.append(p)
            if len(parts) == 1:
                os.replace(parts[0], os.path.join(inc_dir, f'incident_{inc_id}_after.tif'))
                print(f'Planet after ready for incident_{inc_id}')
            elif len(parts) > 1:
                mosaic_geotiffs(parts, os.path.join(inc_dir, f'incident_{inc_id}_after.tif'))
                for p in parts:
                    if os.path.exists(p):
                        os.remove(p)
                print(f'Planet after ready for incident_{inc_id} ({len(parts)} scenes mosaicked)')
            else:
                print(f'Planet after order for incident_{inc_id} has no analytic SR asset yet - skipping')
        except Exception as e:
            print(f'Planet after failed for incident_{inc_id}: {e}')

    # Planet BEFORE download (keep separately if available)
    if before_name in orders_by_name and f'incident_{inc_id}_planet_before.tif' not in have:
        try:
            order = orders_by_name[before_name]
            links = extract_order_asset_links(planet, order, ANALYTIC_SR_SUFFIX)
            parts = []
            for i, link in enumerate(links, 1):
                p = os.path.join(inc_dir, f'_planet_before_part_{i}.tif')
                download_file(planet, link, p)
                parts.append(p)
            outp = os.path.join(inc_dir, f'incident_{inc_id}_planet_before.tif')
            if len(parts) == 1:
                os.replace(parts[0], outp)
                print(f'Planet before ready for incident_{inc_id}')
            elif len(parts) > 1:
                mosaic_geotiffs(parts, outp)
                for p in parts:
                    if os.path.exists(p):
                        os.remove(p)
                print(f'Planet before ready for incident_{inc_id} ({len(parts)} scenes mosaicked)')
            else:
                print(f'Planet before order for incident_{inc_id} has no analytic SR asset yet - skipping')
        except Exception as e:
            print(f'Planet before failed for incident_{inc_id}: {e}')

    # GEE fallback + SAR + DEM (df_sel is already filtered to incidents that have a
    # ready Planet order, so no extra order-presence check is needed here).
    if gee_available:
        mask_s2_clouds, add_aoi_cloud = gee_cloud_helpers()
        aoi = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])
        before_start = (incident_date - pd.DateOffset(days=PRE_DAYS)).strftime('%Y-%m-%d')
        before_end = (incident_date - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
        after_start = (incident_date + pd.DateOffset(days=1)).strftime('%Y-%m-%d')
        after_end = (incident_date + pd.DateOffset(days=POST_DAYS)).strftime('%Y-%m-%d')

        # Each asset below is fetched in its own try/except: a single failure anywhere in
        # this block (e.g. a transient GEE timeout while ranking S2 scenes) should not
        # silently abort every remaining GEE download for the incident.
        if (f'incident_{inc_id}_gee_before.tif' not in have) and (not os.path.exists(os.path.join(inc_dir, f'incident_{inc_id}_gee_before.tif'))):
            try:
                s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                      .filterBounds(aoi)
                      .map(lambda img: add_aoi_cloud(img, aoi))
                      .filter(ee.Filter.lte('aoi_cloud', CLOUD_MAX_AOI)))
                coll = s2.filterDate(before_start, before_end).map(mask_s2_clouds)
                n = coll.size().getInfo()
                if n > 0:
                    imgs = coll.toList(n)
                    best = None
                    best_key = None
                    inc_dt = incident_date
                    if inc_dt.tzinfo is not None:
                        inc_dt = inc_dt.tz_localize(None)
                    for i in range(n):
                        im = ee.Image(imgs.get(i))
                        dt = pd.to_datetime(im.date().format().getInfo())
                        if dt.tzinfo is not None:
                            dt = dt.tz_localize(None)
                        cloud = float(im.get('aoi_cloud').getInfo() or 100)
                        k = (abs((dt - inc_dt).total_seconds()), cloud)
                        if best_key is None or k < best_key:
                            best_key = k
                            best = im
                    bands = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','SCL']
                    gee_download_to_path(best.select(bands).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_gee_before.tif'), scale=10)
                else:
                    print(f'No cloud-free S2 before scene for incident_{inc_id}')
            except Exception as e:
                print(f'GEE before failed for incident_{inc_id}: {e}')

        dem = ee.Image('USGS/SRTMGL1_003')
        if f'incident_{inc_id}_slope.tif' not in have:
            try:
                gee_download_to_path(ee.Terrain.slope(dem).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_slope.tif'), scale=30)
            except Exception as e:
                print(f'GEE slope failed for incident_{inc_id}: {e}')
        if f'incident_{inc_id}_gee_aspect.tif' not in have:
            try:
                gee_download_to_path(ee.Terrain.aspect(dem).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_gee_aspect.tif'), scale=30)
            except Exception as e:
                print(f'GEE aspect failed for incident_{inc_id}: {e}')

        try:
            s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
                  .filterBounds(aoi)
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                  .filter(ee.Filter.eq('instrumentMode', 'IW')))
            pre_coll = s1.filterDate(before_start, before_end)
            post_coll = s1.filterDate(after_start, after_end)
            if f'incident_{inc_id}_sar_pre.tif' not in have and pre_coll.size().getInfo() > 0:
                pre_img = pre_coll.sort('system:time_start', False).first()
                gee_download_to_path(ee.Image(pre_img).select(['VV','VH']).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_sar_pre.tif'), scale=10)
            if f'incident_{inc_id}_sar_post.tif' not in have and post_coll.size().getInfo() > 0:
                post_img = post_coll.sort('system:time_start', True).first()
                gee_download_to_path(ee.Image(post_img).select(['VV','VH']).clip(aoi), aoi, os.path.join(inc_dir, f'incident_{inc_id}_sar_post.tif'), scale=10)
        except Exception as e:
            print(f'GEE SAR failed for incident_{inc_id}: {e}')

    tif_files = sorted(glob.glob(os.path.join(inc_dir, '*.tif')))
    return inc_id, inc_dir, tif_files


download_records = []
pending_ops = []    # list[CommitOperationAdd] waiting for the next batched HF commit
pending_meta = []   # list[(inc_id, n_files, inc_dir)] describing what pending_ops holds


def flush_pending():
    """Commit every currently-queued file in a single batched HF commit, then clean up
    the local incident folders that were just uploaded. Runs only in the main thread."""
    global pending_ops, pending_meta
    if not pending_ops:
        return
    n_incidents = len(pending_meta)
    try:
        hf_api.create_commit(
            repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
            operations=pending_ops,
            commit_message=f'Add raw images for {n_incidents} incident(s)',
        )
        for inc_id, n_files, _ in pending_meta:
            print(f'Uploaded incident_{inc_id}: {n_files} file(s) to HF (batch flush of {len(pending_ops)} files / {n_incidents} incidents)')
            download_records.append({'incident_id': inc_id, 'status': 'uploaded', 'n_files': n_files,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': ''})
    except Exception as e:
        for inc_id, n_files, _ in pending_meta:
            print(f'Upload failed for incident_{inc_id}: {e}')
            download_records.append({'incident_id': inc_id, 'status': 'upload_failed', 'n_files': n_files,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': str(e)})
    finally:
        for _, _, inc_dir in pending_meta:
            shutil.rmtree(inc_dir, ignore_errors=True)
        pending_ops = []
        pending_meta = []


# Skip incidents that already have every mandatory file (and no pending planet_before) up
# front, before even submitting them to the thread pool.
rows_to_process = []
for _, row in df_sel.iterrows():
    inc_id = int(row['id'])
    need = {
        f'incident_{inc_id}_after.tif',
        f'incident_{inc_id}_gee_before.tif',
        f'incident_{inc_id}_sar_pre.tif',
        f'incident_{inc_id}_sar_post.tif',
        f'incident_{inc_id}_slope.tif',
        f'incident_{inc_id}_gee_aspect.tif',
    }
    have = existing_files.get(inc_id, set())
    before_name = f'incident_{inc_id}_planet_before'
    # A Planet 'before' order may finish after the mandatory set is already uploaded;
    # don't permanently skip an incident that still has a pending planet_before to pick up.
    planet_before_pending = (before_name in orders_by_name) and (f'incident_{inc_id}_planet_before.tif' not in have)
    if need.issubset(have) and not planet_before_pending:
        print(f'Skip incident_{inc_id}: all mandatory files already on HF')
        continue
    rows_to_process.append(row)

print(f'Processing {len(rows_to_process)} incident(s) with up to {MAX_WORKERS} worker(s), flushing uploads every {UPLOAD_BATCH_SIZE} file(s)')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_incident, row): int(row['id']) for row in rows_to_process}
    for future in as_completed(futures):
        inc_id = futures[future]
        try:
            _, inc_dir, tif_files = future.result()
        except Exception as e:
            print(f'Incident_{inc_id} processing failed: {e}')
            download_records.append({'incident_id': inc_id, 'status': 'processing_failed', 'n_files': 0,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': str(e)})
            continue

        if not tif_files:
            print(f'No files downloaded for incident_{inc_id} (no ready Planet order and/or no GEE data available yet) - skipping upload')
            download_records.append({'incident_id': inc_id, 'status': 'no_data', 'n_files': 0,
                                      'timestamp': datetime.now(timezone.utc).isoformat(), 'error': 'no source files downloaded'})
            shutil.rmtree(inc_dir, ignore_errors=True)
            continue

        for fp in tif_files:
            pending_ops.append(CommitOperationAdd(path_in_repo=f'{HF_RAW_ROOT}/incident_{inc_id}/{os.path.basename(fp)}', path_or_fileobj=fp))
        pending_meta.append((inc_id, len(tif_files), inc_dir))

        if len(pending_ops) >= UPLOAD_BATCH_SIZE:
            flush_pending()

# Final flush for any incidents left under the batch threshold
flush_pending()

# Upload download log
if len(download_records) > 0:
    dl = pd.DataFrame(download_records)
    local_dl = '/kaggle/working/download_log.csv'
    dl.to_csv(local_dl, index=False)
    hf_api.upload_file(path_or_fileobj=local_dl, path_in_repo=HF_DOWNLOAD_LOG, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, revision=HF_REVISION)

print('Download pass complete')


Skip incident_72660: all mandatory files already on HF
Skip incident_72669: all mandatory files already on HF
Skip incident_72780: all mandatory files already on HF
Skip incident_72638: all mandatory files already on HF
Skip incident_72673: all mandatory files already on HF
Processing 15 incident(s) with up to 2 worker(s), flushing uploads every 25 file(s)


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Planet after ready for incident_72651
No cloud-free S2 before scene for incident_72678
Planet after ready for incident_72619 (2 scenes mosaicked)
Planet after ready for incident_72624


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Uploaded incident_72651: 6 file(s) to HF (batch flush of 30 files / 5 incidents)
Uploaded incident_72627: 6 file(s) to HF (batch flush of 30 files / 5 incidents)
Uploaded incident_72630: 6 file(s) to HF (batch flush of 30 files / 5 incidents)
Uploaded incident_72635: 6 file(s) to HF (batch flush of 30 files / 5 incidents)
Uploaded incident_72637: 6 file(s) to HF (batch flush of 30 files / 5 incidents)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded incident_72578: 6 file(s) to HF (batch flush of 6 files / 1 incidents)
Download pass complete
